In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.data.splitter import Splitter
from src.models.baselines import ApolloPipeline

2026-01-20 12:23:53.713774: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-20 12:23:56.262322: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 1. Configuration & Setup

In [2]:
# Paths
DATA_PATH = "../data/processed/glioma_dataset.h5"
SPLIT_DIR = "../data/splits"
SPLIT_FILE = f"{SPLIT_DIR}/manual_splits.json"

# Ensure Splits Exist
if not os.path.exists(SPLIT_FILE):
    print("Generating Manual Splits from Apollo Paper...")
    splitter = Splitter(DATA_PATH, SPLIT_DIR)
    splitter.generate_manual_split()
else:
    print(f"Loading frozen splits from: {SPLIT_FILE}")

Loading frozen splits from: ../data/splits/manual_splits.json


## 2. Load Data
We use `memory_mapped=True` to keep RAM usage low, although the loader handles this gracefully.

In [3]:
print("Initializing Loader...")
# The loader automatically handles the manual split logic if 'split_file' is provided
loader = GliomaDataLoader(DATA_PATH, split_file=SPLIT_FILE, memory_mapped=True)

Initializing Loader...
Loading data from ../data/processed/glioma_dataset.h5...
Data Loaded. Shape: (294710, 1738)
Unique Samples: 58
Sample ID Example: 'HF-305_v4b_1_1.h5_6' (Type: <class 'numpy.str_'>)


## 3. Run Benchmark (5-Fold CV)

In [ ]:
fold_results = []
feature_masks = []

print("Starting Apollo Baseline Reproduction...")
print("-" * 60)

for fold_data in loader.get_cross_validation_generator():
    fold_id = fold_data['fold']
    print(f"\n>>> Running {fold_id} <<<")
    
    # Unpack Data
    X_train, y_train, _ = fold_data['train']
    X_val, y_val, _ = fold_data['val']
    
    # Print fold stats
    print(f"    Train Samples: {X_train.shape[0]} | Val Samples: {X_val.shape[0]}")
    print(f"    Held-out Patients: {np.unique(fold_data['val_sample_ids'])}")
    
    # Initialize Model
    model = ApolloPipeline(n_features=20)
    
    # Train
    model.fit(X_train, y_train)
    
    # Save the selected features (Critical for Phase 3: TSTR)
    # We will need to force the Synthetic SVM to use THESE SAME FEATURES later.
    feature_masks.append(model.selected_indices_)
    
    # Predict
    y_pred = model.predict(X_val)
    
    # Evaluate
    acc = accuracy_score(y_val, y_pred)
    print(f"    > Fold Accuracy: {acc:.4f}")
    
    # Store detailed results
    fold_results.append({
        'fold': fold_id,
        'accuracy': acc,
        'y_true': y_val,
        'y_pred': y_pred
    })

print("-" * 60)
print("Benchmark Complete.")

Starting Apollo Baseline Reproduction...
------------------------------------------------------------
Loading frozen splits from ../data/splits/manual_splits.json
DEBUG: Split asks for 'HF-2544_V1B_1.h5_3'
DEBUG: Loader has 'HF-305_v4b_1_1.h5_6'

>>> Running fold_0 <<<
    Train Samples: 240035 | Val Samples: 54675
    Held-out Patients: ['HF-1002_V1AT_1.h5_0' 'HF-1002_V2AT_1.h5_1' 'HF-2104_#5_1.h5_0'
 'HF-2104_#9_1.h5_1' 'HF-2104_V1T_1.h5_2' 'HF-305_v4b_1_1.h5_6'
 'HF-442_V4BB_1.h5_12' 'HF-448_V5B_1.h5_3' 'HF-592_V3T_1.h5_4'
 'HF-615_V5BB_1.h5_9' 'HF-682_V3AT_1.h5_9' 'HF-682_V3BB_1.h5_10'
 'HF-894_9_1.h5_11' 'HF-894_V1BB_1.h5_12']
  [Apollo] Fitting RF Selector on (240035, 1738)...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...


## 4. Analysis & Reporting

In [ ]:
# Aggregate Accuracy
accuracies = [r['accuracy'] for r in fold_results]
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

print(f"\n=== Final Results ===")
print(f"Apollo Baseline Accuracy: {mean_acc:.4f} (+/- {std_acc:.4f})")

# Save Baseline Metrics for future comparison
results_df = pd.DataFrame(fold_results)
results_df.to_csv("../results/baseline_metrics.csv", index=False)
print("Saved metrics to ../results/baseline_metrics.csv")

# Save Feature Masks (The 'Golden Features')
np.save("../results/apollo_feature_masks.npy", feature_masks)
print("Saved feature masks to ../results/apollo_feature_masks.npy")

### Confusion Matrix (Aggregated)

In [ ]:
all_true = np.concatenate([r['y_true'] for r in fold_results])
all_pred = np.concatenate([r['y_pred'] for r in fold_results])

cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f"Aggregated Confusion Matrix (Acc: {mean_acc:.4f})")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()